## Nanoparticle Tracking Analysis Pipeline
### Automated detection, tracking, and fluid flow analysis of fluorescent particles in microscopy video data

**Author:** Ben  
**Institution:** Durham University  
**Last Updated:** March 2026

---

#### Pipeline Overview
This notebook provides a complete end-to-end pipeline for analysing particle transport in microscopy videos. It takes a preprocessed `.tif` stack as input and produces quantitative fluid flow maps and statistical analysis outputs.

#### ⚠️ Required Preprocessing (Before Running This Notebook)
Raw `.nd2` videos must be preprocessed in **Fiji/ImageJ** before use:
1. Open the `.nd2` file in Fiji
2. Convert to **16-bit Grayscale** — Image → Type → 16-bit
3. Adjust **Brightness & Contrast** — Image → Adjust → Brightness/Contrast
4. Convert to **8-bit** — Image → Type → 8-bit
5. Export as **TIF stack** — File → Save As → Tiff
6. Note **skin line pixel coordinates** using the line tool — left point `SKIN_P1` and right point `SKIN_P2` — needed for the standardisation step
7. Export **ROI measurements** as a CSV from Fiji/ImageJ if running spatial analysis

> The B&C adjustment is essential to preserve particle visibility before bit-depth conversion, ensuring the YOLO detection model receives optimal input data.

#### Pipeline Steps (This Notebook)
1. Detection — YOLO-based particle detection with tiled inference
2. Tracking — DeepSORT multi-object tracking
3. Stitching — Track cleaning, merging, and gap filling
4. Standardisation — Coordinate alignment to skin reference line
5. Analysis — Velocity maps, stochasticity, capture probability, residence time

#### Requirements
- See `requirements.txt` for dependencies
- A trained YOLO model (`.pt` file)
- Preprocessed `.tif` stack (see preprocessing steps above)
- ROI CSV file (from ImageJ) for spatial analysis steps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# This is the only cell that needs to be edited for each new video.
# All other cells can be run as-is once these values are set.

# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------
TIF_PATH = r"/path/to/your/tif/stack.tif"
MODEL_PATH = r"/path/to/your/trained/model.pt"
ROI_CSV = r"/path/to/your/roi/file.csv"
OUTPUT_DIR = r"/path/to/your/output/directory"
# -----------------------------------------------------------------------------
# VIDEO PARAMETERS
# -----------------------------------------------------------------------------
FPS = 500                    # Frames per second of the video (typically 500 or 1000)
PIXELS_PER_MICRON = 9.2308   # Microscope calibration factor (px/µm)

# -----------------------------------------------------------------------------
# SKIN LINE COORDINATES (pixels)
# From Fiji line tool — left and right points along the skin surface
# -----------------------------------------------------------------------------
SKIN_P1 = (0, 0)
SKIN_P2 = (0, 0)
# -----------------------------------------------------------------------------
# FLOW DIRECTION
# -----------------------------------------------------------------------------
FLIP_DIRECTION = False      # Set to True if flow runs right-to-left

# -----------------------------------------------------------------------------
# DETECTION SETTINGS
# -----------------------------------------------------------------------------
NUM_TILES = 5                # Number of tiles for tiled inference (match training)
TILE_SIZE = 640              # Tile size in pixels (match training — default 640)
CONFIDENCE_THRESHOLD = 0.6   # Minimum detection confidence score (0.0 - 1.0)
NMS_IOU_THRESHOLD = 0.3      # Non-max suppression overlap threshold (0.0 - 1.0)

# -----------------------------------------------------------------------------
# TRACKING SETTINGS
# -----------------------------------------------------------------------------
MAX_AGE = 20                 # Frames to keep a lost track alive before dropping
N_INIT = 10                  # Frames needed to confirm a new track
MAX_COSINE_DISTANCE = 0.05   # Appearance similarity threshold
MAX_IOU_DISTANCE = 0.7       # Spatial overlap threshold
MIN_TRACK_LENGTH = 30        # Minimum frames for a track to be kept (filters noise)

# -----------------------------------------------------------------------------
# ANALYSIS SETTINGS
# -----------------------------------------------------------------------------
SHOW_ROIS = True             # Set to False if no ROI data available
BIN_SIZE = 16                # Spatial bin size for flow maps (pixels)
MAX_SPEED_UM_S = 1500        # Physically plausible upper limit for cilia-driven flow (um/s)
SHOW_QUIVER = True           # Overlay velocity direction arrows on flow maps

In [ ]:
!pip install ultralytics
!pip install deep-sort-realtime

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
# Standard library
import os
import random
import shutil

# Third-party — data handling
import numpy as np
import pandas as pd

# Third-party — image and video processing
import cv2
import tifffile

# Third-party — deep learning and detection
import torch
from ultralytics import YOLO
from torchvision.ops import nms
from deep_sort_realtime.deepsort_tracker import DeepSort

# Third-party — visualisation and analysis
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as patches
import scipy.ndimage as ndimage

# Font settings for publication-quality plots
plt.rcParams["font.family"] = "DejaVu Serif"

print("✅ All libraries imported successfully.")

In [ ]:
# =============================================================================
# STEP 1 — DETECTION
# =============================================================================
# Runs YOLO detection on every frame of the TIF stack using tiled inference.
# Each frame is divided into overlapping tiles to preserve resolution, detections
# are reconstructed to global coordinates, and NMS removes duplicates.
# =============================================================================

def calculate_tile_starts(frame_width, tile_width, num_tiles):
    """
    Calculate evenly distributed tile start positions along the x-axis.

    Parameters:
        frame_width (int): Width of the full frame in pixels.
        tile_width (int): Width of each tile in pixels.
        num_tiles (int): Number of tiles to generate.

    Returns:
        list: X-axis start positions for each tile.
    """
    total_advance_distance = frame_width - tile_width

    if total_advance_distance <= 0 or num_tiles <= 1:
        return [0]

    num_steps = num_tiles - 1
    base_step = total_advance_distance // num_steps
    remainder = total_advance_distance % num_steps

    starts = [0]
    current_start = 0
    for tile_idx in range(1, num_tiles):
        step = base_step + (1 if tile_idx <= remainder else 0)
        current_start += step
        starts.append(current_start)

    return starts


def pad_frame_for_inference(image, target_dim=640):
    """
    Pad image height to match target tile dimension.

    Parameters:
        image (np.ndarray): Input frame.
        target_dim (int): Target square dimension in pixels.

    Returns:
        tuple: (padded_image, top_pad_offset)
    """
    frame_height = image.shape[0]
    h_pad_required = target_dim - frame_height

    if h_pad_required < 0:
        return image, 0

    top_pad = h_pad_required // 2
    bottom_pad = h_pad_required - top_pad

    padded_image = cv2.copyMakeBorder(
        image,
        top_pad, bottom_pad, 0, 0,
        cv2.BORDER_CONSTANT, value=0
    )

    return padded_image, top_pad


def get_inference_tiles(image, tile_dim=640, num_tiles=5):
    """
    Split a padded frame into overlapping tiles for YOLO inference.

    Parameters:
        image (np.ndarray): Input frame.
        tile_dim (int): Tile size in pixels.
        num_tiles (int): Number of tiles.

    Returns:
        tuple: (list of tile images, list of (x_offset, y_offset) tuples)
    """
    padded_img, top_pad = pad_frame_for_inference(image, tile_dim)
    padded_h, padded_w = padded_img.shape[:2]
    x_starts = calculate_tile_starts(padded_w, tile_dim, num_tiles)

    tiles = []
    offsets = []

    for start_x in x_starts:
        tile = padded_img[0:padded_h, start_x:start_x + tile_dim]
        tiles.append(tile)
        offsets.append((start_x, top_pad))

    return tiles, offsets


def preprocess_frame(image):
    """
    Prepare a frame for YOLO inference.
    Converts 16-bit images to 8-bit if necessary, and ensures 3-channel RGB format.

    Parameters:
        image (np.ndarray): Raw input frame.

    Returns:
        np.ndarray: Preprocessed frame ready for YOLO inference.
    """
    if image.dtype == np.uint16:
        image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
        image = image.astype(np.uint8)

    if len(image.shape) == 2:
        image = cv2.merge([image, image, image])
    elif len(image.shape) == 3 and image.shape[2] == 1:
        gray = image[:, :, 0]
        image = cv2.merge([gray, gray, gray])

    return image


def process_frame(frame, model, frame_idx):
    """
    Run full detection pipeline on a single frame.
    Tiles the frame, runs YOLO, reconstructs global coordinates, applies NMS.

    Parameters:
        frame (np.ndarray): Raw video frame.
        model (YOLO): Loaded YOLO model.
        frame_idx (int): Frame index for output labelling.

    Returns:
        list: Detections as dicts with keys: frame, x_topleft, y_topleft, w, h, score, class.
    """
    frame = preprocess_frame(frame)
    tiles, offsets = get_inference_tiles(frame, tile_dim=TILE_SIZE, num_tiles=NUM_TILES)
    results = model(tiles, verbose=False)

    raw_boxes = []
    raw_scores = []
    raw_classes = []

    for tile_idx, result in enumerate(results):
        x_window_start, top_pad_amount = offsets[tile_idx]
        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy()

        for box, score, cls in zip(boxes, scores, classes):
            local_x1, local_y1, local_x2, local_y2 = box

            global_x1 = local_x1 + x_window_start
            global_x2 = local_x2 + x_window_start
            global_y1 = local_y1 - top_pad_amount
            global_y2 = local_y2 - top_pad_amount

            if global_y1 >= 0:
                raw_boxes.append([global_x1, global_y1, global_x2, global_y2])
                raw_scores.append(score)
                raw_classes.append(cls)

    final_detections = []

    if len(raw_boxes) > 0:
        boxes_tensor = torch.tensor(raw_boxes, dtype=torch.float32)
        scores_tensor = torch.tensor(raw_scores, dtype=torch.float32)
        keep_indices = nms(boxes_tensor, scores_tensor, iou_threshold=NMS_IOU_THRESHOLD)

        kept_boxes = boxes_tensor[keep_indices].numpy()
        kept_scores = scores_tensor[keep_indices].numpy()
        kept_classes = np.array(raw_classes)[keep_indices.numpy()]

        for box, score, cls in zip(kept_boxes, kept_scores, kept_classes):
            gx1, gy1, gx2, gy2 = box
            final_detections.append({
                'frame': frame_idx,
                'x_topleft': round(gx1, 2),
                'y_topleft': round(gy1, 2),
                'w': round(gx2 - gx1, 2),
                'h': round(gy2 - gy1, 2),
                'score': round(score, 4),
                'class': int(cls)
            })

    return final_detections


# --- MAIN EXECUTION ---
os.makedirs(OUTPUT_DIR, exist_ok=True)
detection_output_path = os.path.join(OUTPUT_DIR, "detections.csv")

print(f"Loading model: {MODEL_PATH}")
model = YOLO(MODEL_PATH)

print(f"Loading video: {TIF_PATH}")
video_data = tifffile.imread(TIF_PATH)
print(f"   Shape: {video_data.shape}")

all_detections = []
print("\n Running detection...")

for frame_idx, frame in enumerate(video_data):
    if frame_idx % 10 == 0:
        print(f"   Frame {frame_idx} / {len(video_data)}...")
    detections = process_frame(frame, model, frame_idx)
    all_detections.extend(detections)

if len(all_detections) > 0:
    detections_df = pd.DataFrame(all_detections)
    detections_df = detections_df[['frame', 'x_topleft', 'y_topleft', 'w', 'h', 'score', 'class']]
    detections_df.to_csv(detection_output_path, index=False)
    print(f"\n Detection complete. {len(detections_df)} detections saved to: {detection_output_path}")

    MAX_BOX_SIZE = 200  # Maximum bounding box dimension — filters out-of-focus blobs

    detections_df = pd.read_csv(detection_output_path)
    before = len(detections_df)
    detections_df = detections_df[(detections_df['w'] < MAX_BOX_SIZE) & (detections_df['h'] < MAX_BOX_SIZE)]
    after = len(detections_df)
    detections_df.to_csv(detection_output_path, index=False)
    print(f"Bounding box filter applied: {before - after} detections removed ({before} → {after})")

    print(detections_df.head())
else:
    print(" No detections found. Check model path and video.")

In [ ]:
# =============================================================================
# STEP 2 — TRACKING
# =============================================================================
# Runs DeepSORT multi-object tracking on the detection CSV output from Step 1.
# Tracks are confirmed after N_INIT consecutive detections and dropped after
# MAX_AGE frames without a match. Short tracks below MIN_TRACK_LENGTH frames
# are removed as noise in the final cleanup filter.
# =============================================================================

detection_output_path = os.path.join(OUTPUT_DIR, "detections.csv")
tracking_output_path = os.path.join(OUTPUT_DIR, "raw_tracking.csv")
tracking_video_path = os.path.join(OUTPUT_DIR, "raw_tracking.mp4")

# --- LOAD DATA ---
print(f"Loading detections: {detection_output_path}")
detections_df = pd.read_csv(detection_output_path)
detections_df = detections_df[detections_df['score'] >= CONFIDENCE_THRESHOLD]
detections_df = detections_df.sort_values(by='frame')

print(f"Loading video: {TIF_PATH}")
video_stack = tifffile.imread(TIF_PATH)
num_frames, frame_height, frame_width = video_stack.shape

# --- INITIALISE TRACKER ---
tracker = DeepSort(
    max_age=MAX_AGE,
    n_init=N_INIT,
    max_cosine_distance=MAX_COSINE_DISTANCE,
    max_iou_distance=MAX_IOU_DISTANCE,
    embedder_gpu=True
)

# --- INITIALISE VIDEO WRITER ---
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(
    tracking_video_path, fourcc, 30, (frame_width, frame_height)
)

track_history = []

print(f"Running tracking on {num_frames} frames...")

for frame_idx, frame in enumerate(video_stack):

    # Normalise frame to 8-bit RGB for DeepSORT appearance embedding
    if frame.max() > 0:
        frame_8bit = (frame / frame.max() * 255).astype('uint8')
    else:
        frame_8bit = frame.astype('uint8')
    frame_rgb = cv2.cvtColor(frame_8bit, cv2.COLOR_GRAY2BGR)

    # Format detections for DeepSORT: [[left, top, w, h], confidence, class_id]
    frame_detections = detections_df[detections_df['frame'] == frame_idx]
    formatted_detections = []
    for _, row in frame_detections.iterrows():
        formatted_detections.append([
            [row['x_topleft'], row['y_topleft'], row['w'], row['h']],
            row['score'],
            int(row['class'])
        ])

    # Update tracker
    tracks = tracker.update_tracks(formatted_detections, frame=frame_rgb)

    for track in tracks:
        if not track.is_confirmed():
            continue

        ltrb = track.to_ltrb()
        track_history.append({
            'frame': frame_idx,
            'track_id': track.track_id,
            'x': ltrb[0],
            'y': ltrb[1],
            'w': ltrb[2] - ltrb[0],
            'h': ltrb[3] - ltrb[1],
            'center_x': (ltrb[0] + ltrb[2]) / 2,
            'center_y': (ltrb[1] + ltrb[3]) / 2
        })

        # Draw track ID and bounding box on frame
        cv2.putText(
            frame_rgb, str(track.track_id),
            (int(ltrb[0]), int(ltrb[1]) - 10),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2
        )
        cv2.rectangle(
            frame_rgb,
            (int(ltrb[0]), int(ltrb[1])),
            (int(ltrb[2]), int(ltrb[3])),
            (0, 255, 0), 2
        )

    video_writer.write(frame_rgb)

    if frame_idx % 50 == 0:
        print(f"   Frame {frame_idx} / {num_frames}")

video_writer.release()

# --- FINAL CLEANUP FILTER ---
# Remove short tracks below MIN_TRACK_LENGTH as these are likely noise
print(f"\nApplying minimum track length filter (>= {MIN_TRACK_LENGTH} frames)...")
raw_df = pd.DataFrame(track_history)
track_lengths = raw_df.groupby('track_id').size()
valid_track_ids = track_lengths[track_lengths >= MIN_TRACK_LENGTH].index
clean_df = raw_df[raw_df['track_id'].isin(valid_track_ids)]

clean_df.to_csv(tracking_output_path, index=False)

# --- SCOREBOARD ---
print(f"\nTracking complete.")
print(f"Total tracks (raw):     {raw_df['track_id'].nunique()}")
print(f"Total tracks (cleaned): {clean_df['track_id'].nunique()}")
print(f"Tracks removed:         {raw_df['track_id'].nunique() - clean_df['track_id'].nunique()}")
print(f"Average track length:   {clean_df.groupby('track_id').size().mean():.2f} frames")
print(f"Tracking video saved:   {tracking_video_path}")
print(f"Tracking CSV saved:     {tracking_output_path}")

## Step 3 — Stitching

Before running this cell, watch the tracking output video (`raw_tracking.mp4` in your output folder) and identify:

- **Tracks to delete** — spurious detections or noise that got through the length filter
- **Tracks to merge** — the same particle tracked as two separate IDs (broken tracks)
- **Tracks to swap** — two tracks that cross and exchange IDs at a specific frame

Fill in the configuration cell below accordingly, then run both cells.

In [ ]:
# =============================================================================
# STEP 3 — STITCHING CONFIGURATION
# =============================================================================
# Fill these in after reviewing the tracking output video.
# All lists can be left empty if no corrections are needed.

# Track IDs to remove entirely (noise, spurious detections)
DELETE_IDS = []
# Track merges — same particle tracked as two IDs: (keep_id, remove_id)
# For chain merges e.g. 3 tracks into 1: (keep, remove_1), (keep, remove_2)
MERGES = []

# Track swaps — two tracks that cross and swap IDs at a specific frame:
# (id_a, id_b, swap_frame)
SWAPS = []

# Gap filling limits (adjust only if needed — defaults work well in most cases)
MAX_GAP_FRAMES = 10     # Maximum gap in frames to interpolate across
MAX_JUMP_PIXELS = 20.0  # Maximum pixel distance to allow interpolation

In [ ]:
# =============================================================================
# STEP 3 — STITCHING
# =============================================================================
# Cleans and repairs the raw tracking output by:
# 1. Removing spurious tracks (DELETE_IDS)
# 2. Merging broken tracks (MERGES)
# 3. Correcting ID swaps at crossing points (SWAPS)
# 4. Interpolating small gaps using distance-aware linear interpolation
# =============================================================================

stitching_input_path = os.path.join(OUTPUT_DIR, "raw_tracking.csv")
stitching_output_path = os.path.join(OUTPUT_DIR, "stitched_tracking.csv")


def run_stitching(input_path, output_path):
    """
    Clean and repair raw tracking data.

    Parameters:
        input_path (str): Path to raw tracking CSV.
        output_path (str): Path to save stitched tracking CSV.
    """
    print(f"Loading: {input_path}")
    df = pd.read_csv(input_path)

    # --- PHASE 1: DELETE SPURIOUS TRACKS ---
    if DELETE_IDS:
        df = df[~df['track_id'].isin(DELETE_IDS)]
        print(f"Deleted {len(DELETE_IDS)} spurious tracks.")

    # --- PHASE 2: SWAPS ---
    # Corrects cases where two tracks cross and exchange IDs at a specific frame
    if SWAPS:
        for id_a, id_b, swap_frame in SWAPS:
            mask_a = (df['track_id'] == id_a) & (df['frame'] >= swap_frame)
            mask_b = (df['track_id'] == id_b) & (df['frame'] >= swap_frame)
            df.loc[mask_a, 'temp_swap'] = id_b
            df.loc[mask_b, 'temp_swap'] = id_a
            valid_swap_mask = df['temp_swap'].notna()
            df.loc[valid_swap_mask, 'track_id'] = df.loc[valid_swap_mask, 'temp_swap'].astype(int)
        if 'temp_swap' in df.columns:
            df = df.drop(columns=['temp_swap'])
        print(f"Applied {len(SWAPS)} track swaps.")

    # --- PHASE 3: MERGES ---
    # Combines broken tracks that belong to the same particle
    if MERGES:
        for keep_id, remove_id in MERGES:
            if remove_id not in df['track_id'].values:
                print(f"Note: Track {remove_id} not found, skipping.")
                continue
            df.loc[df['track_id'] == remove_id, 'track_id'] = keep_id
        print(f"Applied {len(MERGES)} track merges.")

    # --- PHASE 4: DISTANCE-AWARE GAP FILLING ---
    # Interpolates small gaps between detections, but only if the distance
    # across the gap is below MAX_JUMP_PIXELS — prevents false connections
    df = df.sort_values(by=['track_id', 'frame'])
    print(f"Stitching gaps (max {MAX_GAP_FRAMES} frames, max {MAX_JUMP_PIXELS} px)...")

    def smart_interpolate(group):
        """
        Interpolate gaps within a single track using distance-aware logic.

        Parameters:
            group (pd.DataFrame): Single track data grouped by track_id.

        Returns:
            pd.DataFrame: Track with gaps filled where safe to do so.
        """
        # Remove duplicate frames from overlapping merges
        group = group.drop_duplicates(subset=['frame'], keep='first')

        # Reindex to full frame range to expose gaps
        min_frame = group['frame'].min()
        max_frame = group['frame'].max()
        full_index = np.arange(min_frame, max_frame + 1)
        group = group.set_index('frame').reindex(full_index)

        # Calculate distance across each gap before filling
        # ffill gives the value at the start of the gap
        # bfill gives the value at the end of the gap
        start_vals = group[['x', 'y']].ffill(limit=MAX_GAP_FRAMES)
        end_vals = group[['x', 'y']].bfill(limit=MAX_GAP_FRAMES)
        gap_distance = np.sqrt(
            (start_vals['x'] - end_vals['x'])**2 +
            (start_vals['y'] - end_vals['y'])**2
        )

        # Only interpolate gaps where the distance is small enough
        safe_mask = (group['x'].notna()) | (gap_distance < MAX_JUMP_PIXELS)

        coordinate_cols = ['x', 'y', 'w', 'h', 'center_x', 'center_y']
        interpolated = group[coordinate_cols].interpolate(
            method='linear', limit=MAX_GAP_FRAMES
        )
        group.loc[safe_mask, coordinate_cols] = interpolated.loc[safe_mask]

        # Always fill track_id to keep track connected in data
        group['track_id'] = group['track_id'].ffill().bfill()

        # Drop rows with no track_id (large gaps that exceeded the frame limit)
        group = group.dropna(subset=['track_id'])

        return group.reset_index().rename(columns={'index': 'frame'})

    df_clean = df.groupby('track_id').apply(
        smart_interpolate
    ).reset_index(drop=True)

    df_clean.to_csv(output_path, index=False)
    print(f"Stitching complete. {df_clean['track_id'].nunique()} final tracks.")
    print(f"Saved to: {output_path}")


run_stitching(stitching_input_path, stitching_output_path)

In [ ]:
# =============================================================================
# STEP 3 — STITCHING VERIFICATION
# =============================================================================
# Generates a spaghetti map and video of all stitched tracks overlaid on the
# raw video. Review both outputs before proceeding to Step 4 — if tracks look
# incorrect, adjust DELETE_IDS, MERGES, or SWAPS and re-run stitching.
# =============================================================================

spaghetti_video_path = os.path.join(OUTPUT_DIR, "stitched_spaghetti.mp4")
spaghetti_map_path   = os.path.join(OUTPUT_DIR, "stitched_spaghetti_map.png")

MAX_DRAW_DISTANCE = 50.0  # Maximum pixel distance between consecutive points to draw

# --- LOAD DATA ---
print(f"Loading: {stitching_output_path}")
df_stitch = pd.read_csv(stitching_output_path)
df_stitch = df_stitch.sort_values(by=['frame', 'track_id'])

print(f"Loading video: {TIF_PATH}")
video_stack = tifffile.imread(TIF_PATH)
num_frames, height, width = video_stack.shape

# --- ASSIGN RANDOM COLOURS PER TRACK ---
unique_ids = df_stitch['track_id'].unique()
random.seed(42)
colors = {
    uid: (random.randint(50, 255), random.randint(50, 255), random.randint(50, 255))
    for uid in unique_ids
}

# --- GENERATE SPAGHETTI VIDEO ---
fourcc       = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(spaghetti_video_path, fourcc, 30, (width, height))
track_history = {}

print(f"Generating spaghetti video ({num_frames} frames)...")

for frame_idx in range(num_frames):
    frame      = video_stack[frame_idx]
    frame_norm = (frame / frame.max() * 255).astype('uint8') if frame.max() > 0 else frame.astype('uint8')
    frame_rgb  = cv2.cvtColor(frame_norm, cv2.COLOR_GRAY2BGR)

    current_frame_data = df_stitch[df_stitch['frame'] == frame_idx]

    for _, row in current_frame_data.iterrows():
        if pd.isna(row['x']):
            continue

        tid = int(row['track_id'])
        cx, cy = int(row['center_x']), int(row['center_y'])
        x, y   = int(row['x']), int(row['y'])
        w, h   = int(row['w']), int(row['h'])

        if tid not in track_history:
            track_history[tid] = []
        track_history[tid].append((frame_idx, cx, cy))

        # Draw trail — stop at gaps or jumps
        history = track_history[tid]
        if len(history) > 1:
            for k in range(len(history) - 1, 0, -1):
                curr_f, curr_x, curr_y = history[k]
                prev_f, prev_x, prev_y = history[k - 1]
                time_gap = curr_f - prev_f
                dist     = np.sqrt((curr_x - prev_x)**2 + (curr_y - prev_y)**2)
                if time_gap == 1 and dist < MAX_DRAW_DISTANCE:
                    cv2.line(frame_rgb, (prev_x, prev_y), (curr_x, curr_y), colors[tid], 2)
                else:
                    break

        cv2.rectangle(frame_rgb, (x, y), (x + w, y + h), colors[tid], 2)
        cv2.putText(frame_rgb, str(tid), (x, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

    cv2.putText(frame_rgb, f"Frame: {frame_idx}", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    video_writer.write(frame_rgb)

    if frame_idx % 100 == 0:
        print(f"   Frame {frame_idx} / {num_frames}")

video_writer.release()
print(f"Spaghetti video saved: {spaghetti_video_path}")

# --- GENERATE SPAGHETTI MAP ---
master_map = np.zeros((height, width, 3), dtype='uint8')

for tid in unique_ids:
    track_data  = df_stitch[df_stitch['track_id'] == tid].sort_values('frame')
    prev_point  = None
    prev_frame  = -999

    for _, row in track_data.iterrows():
        if pd.isna(row['center_x']):
            continue

        curr_point = (int(row['center_x']), int(row['center_y']))
        curr_frame = row['frame']

        if prev_point is not None:
            time_gap = curr_frame - prev_frame
            dist     = np.sqrt((curr_point[0] - prev_point[0])**2 +
                               (curr_point[1] - prev_point[1])**2)
            if time_gap == 1 and dist < MAX_DRAW_DISTANCE:
                cv2.line(master_map, prev_point, curr_point, colors[tid], 2)

        prev_point = curr_point
        prev_frame = curr_frame

    if prev_point is not None:
        cv2.putText(master_map, str(tid), prev_point,
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, colors[tid], 2)

cv2.imwrite(spaghetti_map_path, master_map)
print(f"Spaghetti map saved: {spaghetti_map_path}")
print(f"\nReview both outputs before proceeding to Step 4.")
print(f"If tracks look incorrect, adjust stitching config and re-run Step 3.")

## Step 4 — Standardisation

Rotates and aligns all track coordinates relative to the skin surface line,
so that:
- `center_x` represents distance along the skin (µm)
- `center_y` represents height above the skin (µm, 0 = skin surface)
- Flow direction is standardised to left-to-right

Requires `SKIN_P1`, `SKIN_P2` and `FLIP_DIRECTION` set in the Configuration cell.
A verification plot will display showing the skin line overlaid on the first frame —
confirm this looks correct before proceeding.

In [ ]:
# =============================================================================
# STEP 4 — STANDARDISATION
# =============================================================================
# Rotates track coordinates to align with the skin surface reference line.
# Produces a standardised coordinate system where center_y represents height
# above the skin and center_x represents distance along the skin.
# A verification plot is shown before saving to allow visual confirmation.
# =============================================================================

standardisation_input_path = os.path.join(OUTPUT_DIR, "stitched_tracking.csv")
standardisation_output_path = os.path.join(OUTPUT_DIR, "standardised_tracking.csv")


def show_skin_line_verification(tif_path, p1, p2):
    """
    Display the first frame of the video with the proposed skin line overlaid.
    Allows visual confirmation that skin coordinates are correct before
    applying the standardisation transformation.

    Parameters:
        tif_path (str): Path to the TIF stack.
        p1 (tuple): Left skin line coordinate (x, y) in pixels.
        p2 (tuple): Right skin line coordinate (x, y) in pixels.
    """
    try:
        frame = tifffile.imread(tif_path, key=0)
        fig, ax = plt.subplots(figsize=(12, 7))
        ax.imshow(frame, cmap='gray')
        ax.plot([p1[0], p2[0]], [p1[1], p2[1]],
                color='cyan', linewidth=2, label='Skin line')
        ax.scatter([p1[0], p2[0]], [p1[1], p2[1]], color='red', s=40)
        ax.set_title("Verify skin line before proceeding")
        ax.legend()
        ax.axis('off')
        plt.show()
    except Exception as e:
        print(f"Could not load TIF for verification: {e}")


def standardise_tracks(df, p1, p2, flip_x=False):
    """
    Rotate and align track coordinates to the skin surface reference line.

    Parameters:
        df (pd.DataFrame): Stitched tracking data.
        p1 (tuple): Left skin line coordinate (x, y) in pixels.
        p2 (tuple): Right skin line coordinate (x, y) in pixels.
        flip_x (bool): If True, mirror x-axis for right-to-left flow videos.

    Returns:
        tuple: (standardised DataFrame, rotation angle in radians)
    """
    # Calculate skin line tilt angle
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    angle_rad = np.arctan2(dy, dx)
    cos_a = np.cos(-angle_rad)
    sin_a = np.sin(-angle_rad)

    # Rotate center coordinates relative to P1
    tx_c = df['center_x'] - p1[0]
    ty_c = df['center_y'] - p1[1]
    df['center_x'] = (tx_c * cos_a - ty_c * sin_a)

    # Invert y-axis so that higher center_y = further from skin
    rotated_y = (tx_c * sin_a + ty_c * cos_a)
    df['center_y'] = rotated_y.max() - rotated_y

    # Mirror x-axis for right-to-left flow videos
    if flip_x:
        print("Applying right-to-left flow correction...")
        x_max_before_flip = df['center_x'].max()
        df['center_x'] = x_max_before_flip - df['center_x']

    # Reconstruct bounding boxes from standardised centers
    df['x'] = df['center_x'] - (df['w'] / 2)
    df['y'] = df['center_y'] - (df['h'] / 2)


    return df, angle_rad, x_max_before_flip if flip_x else df['center_x'].max()


# --- LOAD DATA ---
print(f"Loading: {standardisation_input_path}")
tracks_df = pd.read_csv(standardisation_input_path)

# --- VERIFICATION PLOT ---
show_skin_line_verification(TIF_PATH, SKIN_P1, SKIN_P2)

# --- APPLY STANDARDISATION ---
standardised_df, skin_angle, std_x_max = standardise_tracks(
    tracks_df, SKIN_P1, SKIN_P2, flip_x=FLIP_DIRECTION
)

# --- SAVE ---
standardised_df.to_csv(standardisation_output_path, index=False)

print(f"Standardisation complete.")
print(f"Skin angle: {np.degrees(skin_angle):.2f} degrees")
print(f"Saved to: {standardisation_output_path}")

## Step 5 — Analysis

Generates quantitative fluid flow maps and statistical analysis outputs from the
standardised tracking data. Requires `SHOW_ROIS = True` and a valid `ROI_CSV` path
in the Configuration cell for spatial analysis overlays.

> If no ROI data is available for this video, set `SHOW_ROIS = False` in the
> Configuration cell — all other outputs will still be generated.

In [ ]:
# =============================================================================
# STEP 5A — VELOCITY AND STOCHASTICITY MAPS
# =============================================================================
# Generates spatial bin maps of:
# - Mean velocity magnitude across the field of view
# - Velocity stochasticity (frame-to-frame fluctuations)
# Both maps include streamlines and velocity direction arrows.
# ROI overlays are drawn if SHOW_ROIS is enabled in configuration.
# =============================================================================

analysis_input_path = os.path.join(OUTPUT_DIR, "standardised_tracking.csv")
video_name = os.path.basename(OUTPUT_DIR)

# --- LOAD AND PREPARE DATA ---
print(f"Loading: {analysis_input_path}")
df = pd.read_csv(analysis_input_path)

# Remove any manually identified bad tracks
BAD_TRACKS = []  # TODO: Add any track IDs to remove from analysis only
df = df[~df['track_id'].isin(BAD_TRACKS)]

# Calculate frame-to-frame velocity components
df = df.sort_values(['track_id', 'frame'])
df['dx'] = df.groupby('track_id')['center_x'].diff()
df['dy'] = df.groupby('track_id')['center_y'].diff()
df['vx_um_s'] = (df['dx'] / PIXELS_PER_MICRON) * FPS
df['vy_um_s'] = (df['dy'] / PIXELS_PER_MICRON) * FPS
df = df.dropna(subset=['vx_um_s', 'vy_um_s'])

# Spatial extent of track data
v_width  = df['center_x'].max()
v_height = df['center_y'].max()
v_min_x  = df['center_x'].min()
v_min_y  = df['center_y'].min()

# --- ROI TRANSFORMATION ---
if SHOW_ROIS:
    dx_skin = SKIN_P2[0] - SKIN_P1[0]
    dy_skin = SKIN_P2[1] - SKIN_P1[1]
    tilt_angle = np.arctan2(dy_skin, dx_skin)
    cos_a = np.cos(-tilt_angle)
    sin_a = np.sin(-tilt_angle)

    def standardise_roi(row, p1):
        """
        Transform ROI coordinates to match the standardised track coordinate system.

        Parameters:
            row (pd.Series): Single ROI row with BX, BY, Width, Height in microns.
            p1 (tuple): Left skin line coordinate (x, y) in pixels.

        Returns:
            pd.Series: Standardised ROI with std_x, std_y, w_px, h_px.
        """
        # Convert ROI centre to pixel space for rotation
        raw_x = row['BX'] * PIXELS_PER_MICRON + (row['Width'] * PIXELS_PER_MICRON / 2)
        raw_y = row['BY'] * PIXELS_PER_MICRON + (row['Height'] * PIXELS_PER_MICRON / 2)

        tx = raw_x - p1[0]
        ty = raw_y - p1[1]

        std_x = tx * cos_a - ty * sin_a
        std_y = tx * sin_a + ty * cos_a

        roi_width_px  = row['Width'] * PIXELS_PER_MICRON
        roi_height_px = row['Height'] * PIXELS_PER_MICRON

        std_x = std_x - roi_width_px / 2
        std_y = -std_y - roi_height_px / 2

        if FLIP_DIRECTION:
            std_x = std_x_max - std_x - roi_width_px


        return pd.Series([std_x, std_y, roi_width_px, roi_height_px])

    roi_raw = pd.read_csv(ROI_CSV)
    roi_df = roi_raw.apply(lambda row: standardise_roi(row, SKIN_P1), axis=1)
    roi_df.columns = ['std_x', 'std_y', 'w_px', 'h_px']

# --- PIV-STYLE SPATIAL BINNING ---
MIN_DETECTIONS = 3
x_bins = np.arange(v_min_x, v_width,  BIN_SIZE)
y_bins = np.arange(v_min_y, v_height, BIN_SIZE)

mean_vx = np.full((len(y_bins), len(x_bins)), np.nan)
mean_vy = np.full((len(y_bins), len(x_bins)), np.nan)
std_v   = np.full((len(y_bins), len(x_bins)), np.nan)

for y_idx, y_start in enumerate(y_bins):
    for x_idx, x_start in enumerate(x_bins):
        bin_mask = (
            (df['center_x'] >= x_start) & (df['center_x'] < x_start + BIN_SIZE) &
            (df['center_y'] >= y_start) & (df['center_y'] < y_start + BIN_SIZE)
        )
        bin_data = df[bin_mask]

        if len(bin_data) < MIN_DETECTIONS:
            continue

        frame_groups = bin_data.groupby('frame')
        frame_mean_vx    = frame_groups['vx_um_s'].mean()
        frame_mean_vy    = frame_groups['vy_um_s'].mean()
        frame_mean_speed = np.sqrt(frame_mean_vx**2 + frame_mean_vy**2)

        mean_vx[y_idx, x_idx] = frame_mean_vx.mean()
        mean_vy[y_idx, x_idx] = frame_mean_vy.mean()
        std_v[y_idx, x_idx]   = frame_mean_speed.std()

# --- SMOOTHING ---
def smooth_nan_aware(arr, sigma):
    """
    Apply Gaussian smoothing to an array containing NaN values.
    NaN regions are handled using a weighted smoothing approach.

    Parameters:
        arr (np.ndarray): Input array with possible NaN values.
        sigma (float): Gaussian smoothing radius.

    Returns:
        np.ndarray: Smoothed array with NaN regions preserved.
    """
    filled = arr.copy()
    filled[np.isnan(filled)] = 0.0
    smoothed = ndimage.gaussian_filter(filled, sigma=sigma)
    weight = (~np.isnan(arr)).astype(float)
    smooth_weight = ndimage.gaussian_filter(weight, sigma=sigma)
    with np.errstate(invalid='ignore'):
        result = np.where(smooth_weight > 0, smoothed / smooth_weight, np.nan)
    return result

SMOOTHING_SIGMA = 1.2
smooth_std   = smooth_nan_aware(std_v,                               SMOOTHING_SIGMA)
smooth_speed = smooth_nan_aware(np.sqrt(mean_vx**2 + mean_vy**2),   SMOOTHING_SIGMA)
smooth_vx    = smooth_nan_aware(mean_vx,                             SMOOTHING_SIGMA)
smooth_vy    = smooth_nan_aware(mean_vy,                             SMOOTHING_SIGMA)

# Convert spatial extents to microns for plot axes
x_min_um = v_min_x / PIXELS_PER_MICRON
x_max_um = v_width  / PIXELS_PER_MICRON
y_min_um = v_min_y  / PIXELS_PER_MICRON
y_max_um = v_height / PIXELS_PER_MICRON

x_range_um = np.linspace(x_min_um, x_max_um, mean_vx.shape[1])
y_range_um = np.linspace(y_min_um, y_max_um, mean_vx.shape[0])

# --- GENERATE AND SAVE PLOTS ---
plot_configs = [
    (smooth_std,   f"{video_name}_stochastic_variation_map.png",
     'Velocity Fluctuations ($\\mu$m/s)', 'magma'),
    (smooth_speed, f"{video_name}_mean_velocity_map.png",
     'Mean Velocity Magnitude ($\\mu$m/s)', 'gist_earth')
]

for data_to_plot, filename, legend_label, cmap_choice in plot_configs:
    fig, ax = plt.subplots(figsize=(16, 8), facecolor='white')
    cmap_obj = mpl.colormaps[cmap_choice]
    ax.set_facecolor(cmap_obj(0))

    im = ax.imshow(
        data_to_plot,
        extent=[x_min_um, x_max_um, y_min_um, y_max_um],
        cmap=cmap_obj, alpha=1.0, origin='lower', interpolation='bilinear'
    )

    # Streamlines showing flow direction
    vx_plot = np.nan_to_num(smooth_vx, nan=0.0)
    vy_plot = np.nan_to_num(smooth_vy, nan=0.0)
    ax.streamplot(
        x_range_um, y_range_um, vx_plot, vy_plot,
        color='#ffffff33', linewidth=0.8, density=1.2,
        arrowsize=0, broken_streamlines=False
    )

    # Velocity direction arrows
    if SHOW_QUIVER:
        X, Y = np.meshgrid(x_range_um, y_range_um)
        skip = (slice(None, None, 3), slice(None, None, 3))
        u = smooth_vx[skip]
        v = smooth_vy[skip]
        magnitude = np.sqrt(u**2 + v**2) + 1e-10  # Small offset prevents division by zero
        ax.quiver(
            X[skip], Y[skip],
            u / magnitude, v / magnitude,
            color='white', alpha=0.5,
            pivot='mid', scale=80,
            width=0.001, headwidth=3, zorder=4
        )

    # ROI overlays
    if SHOW_ROIS:
        for roi_idx, roi_row in roi_df.iterrows():
            rx_um = roi_row['std_x'] / PIXELS_PER_MICRON
            ry_um = roi_row['std_y'] / PIXELS_PER_MICRON
            rw_um = roi_row['w_px'] / PIXELS_PER_MICRON
            rh_um = roi_row['h_px'] / PIXELS_PER_MICRON
            rect = patches.Rectangle(
                (rx_um, ry_um), rw_um, rh_um,
                linewidth=2, edgecolor='cyan', facecolor='none', zorder=5
            )
            ax.add_patch(rect)
            roi_label = len(roi_df) - roi_idx if FLIP_DIRECTION else roi_idx + 1
            ax.text(rx_um + 1, ry_um + 3, f"B{roi_label}",
                color='cyan', weight='bold', fontsize=16, zorder=6)

    cbar = fig.colorbar(im, ax=ax, orientation='horizontal', location='top', fraction=0.04, pad=0.02, aspect=40)
    cbar.set_label(legend_label, fontsize=18, fontfamily='serif')
    ax.set_xlabel("Distance along skin ($\\mu$m)", fontsize=18)
    ax.set_ylabel("Height above skin ($\\mu$m)", fontsize=18)
    ax.tick_params(axis='both', labelsize=16)
    cbar.ax.tick_params(labelsize=16)
    ax.set_xlim(x_min_um, x_max_um)
    ax.set_ylim(y_min_um, y_max_um)

    save_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {save_path}")

In [ ]:
# =============================================================================
# STEP 5B — CAPTURE PROBABILITY
# =============================================================================
# Calculates the probability of a particle being captured by each cilia bundle
# ROI based on its height above the skin at a gate point 40um upstream.
# Results are binned in 5um height bins and saved as a CSV table.
# NOTE: Requires SHOW_ROIS = True and a valid ROI_CSV in configuration.
# =============================================================================

capture_output_path = os.path.join(OUTPUT_DIR, f"{video_name}_capture_probability.csv")


def calculate_capture_probability(tracks, rois, buffer_um=20):
    """
    Calculate the probability of particle capture by each cilia bundle ROI.
    For each ROI, particles are observed at a gate point upstream and tracked
    to determine whether they subsequently enter the ROI.

    Parameters:
        tracks (pd.DataFrame): Standardised tracking data.
        rois (pd.DataFrame): Standardised ROI data with std_x, std_y, w_px, h_px.
        buffer_um (float): Distance upstream of ROI to observe particles (um).

    Returns:
        pd.DataFrame: Capture results with height_um and captured columns.
    """
    gate_distance_px = buffer_um * PIXELS_PER_MICRON
    results = []

    for _, roi_row in rois.iterrows():
        rx = roi_row['std_x']
        ry = roi_row['std_y']
        rw = roi_row['w_px']
        rh = roi_row['h_px']

        # Gate position — capped at 0 to handle ROIs near the left edge
        gate_x = max(0, rx - gate_distance_px)

        # Find all tracks that passed through the gate
        potential_track_ids = tracks[
            tracks['center_x'] <= gate_x
        ]['track_id'].unique()

        for track_id in potential_track_ids:
            track = tracks[tracks['track_id'] == track_id]
            try:
                # Record particle height at the gate point
                gate_point = track[track['center_x'] <= gate_x].iloc[-1]
                height_um = max(0, gate_point['center_y'] / PIXELS_PER_MICRON)

                # Check if particle subsequently entered the ROI
                entered_roi = any(
                    (track['center_x'] >= rx) & (track['center_x'] <= rx + rw) &
                    (track['center_y'] >= ry) & (track['center_y'] <= ry + rh)
                )

                results.append({
                    'height_um': height_um,
                    'captured': entered_roi
                })

            except IndexError:
                continue

    return pd.DataFrame(results)


if SHOW_ROIS:
    capture_df = calculate_capture_probability(df, roi_df)

    # Bin results into 5um height bins
    capture_df['bin'] = (capture_df['height_um'] // 5) * 5

    prob_table = capture_df.groupby('bin')['captured'].agg(
        ['count', 'sum', 'mean']
    )
    prob_table['mean'] *= 100
    prob_table = prob_table.rename(columns={
        'count': 'N',
        'sum': 'N_captured',
        'mean': 'Capture %'
    })

    # Exclude bins below minimum detection threshold
    prob_table.loc[prob_table['N'] < MIN_DETECTIONS, 'Capture %'] = np.nan

    # Force full range 0 to 50um — empty bins stay as NaN for correct aggregation
    full_bin_range = np.arange(0.0, 55.0, 5.0)
    prob_table = prob_table.reindex(full_bin_range)
    prob_table.index.name = 'Height Bin (um from Skin)'

    prob_table.to_csv(capture_output_path)
    print(f"Capture probability analysis complete.")
    print(f"Saved: {capture_output_path}")
    print(prob_table)

else:
    print("SHOW_ROIS is False — skipping capture probability analysis.")

In [ ]:
# =============================================================================
# STEP 5C — RESIDENCE TIME
# =============================================================================
# Calculates the average time particles spend inside each cilia bundle ROI,
# binned by height above the skin in 5um intervals.
# Results are saved as a CSV table in both frames and seconds.
# NOTE: Requires SHOW_ROIS = True and a valid ROI_CSV in configuration.
# =============================================================================

residence_output_path = os.path.join(OUTPUT_DIR, f"{video_name}_residence_times.csv")


def calculate_residence_time(tracks, rois, buffer_um=20):
    """
    Calculate the average time particles spend inside each cilia bundle ROI.
    Only particles that actually enter the ROI are included in the results.

    Parameters:
        tracks (pd.DataFrame): Standardised tracking data.
        rois (pd.DataFrame): Standardised ROI data with std_x, std_y, w_px, h_px.
        buffer_um (float): Distance upstream of ROI to observe particles (um).

    Returns:
        pd.DataFrame: Residence results with height_um and frames columns.
    """
    gate_distance_px = buffer_um * PIXELS_PER_MICRON
    results = []

    for _, roi_row in rois.iterrows():
        rx = roi_row['std_x']
        ry = roi_row['std_y']
        rw = roi_row['w_px']
        rh = roi_row['h_px']

        # Gate position — capped at 0 to handle ROIs near the left edge
        gate_x = max(0, rx - gate_distance_px)

        # Find all tracks that passed through the gate
        potential_track_ids = tracks[
            tracks['center_x'] <= gate_x
        ]['track_id'].unique()

        for track_id in potential_track_ids:
            track = tracks[tracks['track_id'] == track_id]
            try:
                # Record particle height at the gate point
                gate_point = track[track['center_x'] <= gate_x].iloc[-1]
                height_um = max(0, gate_point['center_y'] / PIXELS_PER_MICRON)

                # Count frames spent inside the ROI
                inside_roi_mask = (
                    (track['center_x'] >= rx) & (track['center_x'] <= rx + rw) &
                    (track['center_y'] >= ry) & (track['center_y'] <= ry + rh)
                )
                frames_inside = inside_roi_mask.sum()

                if frames_inside > 0:
                    results.append({
                        'height_um': height_um,
                        'frames': frames_inside
                    })

            except IndexError:
                continue

    return pd.DataFrame(results)


if SHOW_ROIS:
    residence_df = calculate_residence_time(df, roi_df)

    # Bin results into 5um height bins
    residence_df['bin'] = (residence_df['height_um'] // 4) * 4

    # Aggregate by bin
    res_table = residence_df.groupby('bin').agg(
        N      = ('frames', 'count'),
        frames = ('frames', 'mean')
    )

    # Exclude bins below minimum detection threshold
    res_table.loc[res_table['N'] < MIN_DETECTIONS, 'frames'] = np.nan

    # Force full range 0 to 50um — empty bins stay as NaN for correct aggregation
    full_bin_range = np.arange(0.0, 52.0, 4.0)
    res_table = res_table.reindex(full_bin_range)

    # Add seconds column
    res_table['Avg Residence (s)'] = res_table['frames'] / FPS

    # Clean up column names
    res_table.columns    = ['N', 'Avg Residence (Frames)', 'Avg Residence (s)']
    res_table.index.name = "Height Bin (um from Skin)"
    res_table            = res_table.round(4)


    res_table.to_csv(residence_output_path)
    print(f"Residence time analysis complete.")
    print(f"Saved: {residence_output_path}")
    print(res_table)

else:
    print("SHOW_ROIS is False — skipping residence time analysis.")

In [ ]:
# =============================================================================
# STEP 5D — AVERAGE SPEED VS HEIGHT
# =============================================================================
# Calculates mean particle speed binned by height above the skin surface.
# Speed is derived from frame-to-frame velocity components computed in Step 5A.
# Bins with fewer than MIN_DETECTIONS are excluded as unreliable.
# Empty bins stored as NaN to allow correct weighted aggregation across videos.
# =============================================================================

speed_output_path = os.path.join(OUTPUT_DIR, f"{video_name}_average_speed_vs_height.csv")

# --- SPEED CALCULATION ---
df['speed_um_s'] = np.sqrt(df['vx_um_s']**2 + df['vy_um_s']**2)

# Cap physically implausible speeds — values above MAX_SPEED_UM_S are considered
# biologically unrealistic and are excluded from analysis
df_speed = df[df['speed_um_s'] <= MAX_SPEED_UM_S].copy()

# --- HEIGHT BINNING & AGGREGATION ---
df_speed['height_um'] = df_speed['center_y'] / PIXELS_PER_MICRON
df_speed['bin']       = (df_speed['height_um'] // 5) * 5

speed_table = df_speed.groupby('bin').agg(
    N             = ('speed_um_s', 'count'),
    avg_speed     = ('speed_um_s', 'mean')
)

# Exclude bins below minimum detection threshold
speed_table.loc[speed_table['N'] < MIN_DETECTIONS, 'avg_speed'] = np.nan

# Force full range 0 to 50um — empty bins stay as NaN for correct aggregation
full_bin_range = np.arange(0.0, 55.0, 5.0)
speed_table = speed_table.reindex(full_bin_range)

speed_table.columns    = ['N', 'Avg Speed (um/s)']
speed_table.index.name = 'Height Bin (um from Skin)'
speed_table            = speed_table.round(4)

speed_table.to_csv(speed_output_path)
print(f"Average speed vs height analysis complete.")
print(f"Saved: {speed_output_path}")
print(speed_table)

In [ ]:
# =============================================================================
# STEP 5E — ROI ENTRY/EXIT HEIGHT ANALYSIS
# =============================================================================
# For each particle that passes through a cilia bundle ROI, records the height
# above the skin at entry and exit. Calculates mean exit height and mean height
# change per 2um entry height bin, pooled across all ROIs in the video.
# Each ROI interaction is treated as an independent event.
# Empty bins stored as NaN to allow correct weighted aggregation across videos.
# NOTE: Requires SHOW_ROIS = True and a valid ROI_CSV in configuration.
# =============================================================================

entry_exit_output_path = os.path.join(OUTPUT_DIR, f"{video_name}_roi_entry_exit_heights.csv")

BIN_SIZE_UM = 4  # 4um bins for fine resolution within ROI height range

def calculate_entry_exit_heights(tracks, rois):
    """
    For each particle passing through each ROI, record entry and exit heights.
    Entry is the first frame inside the ROI, exit is the last frame inside.
    Each ROI interaction is treated as an independent event — a particle
    entering multiple ROIs contributes one event per ROI.

    Parameters:
        tracks (pd.DataFrame): Standardised tracking data.
        rois (pd.DataFrame): Standardised ROI data with std_x, std_y, w_px, h_px.

    Returns:
        pd.DataFrame: Results with entry_height_um, exit_height_um, height_change_um.
    """
    results = []

    for _, roi_row in rois.iterrows():
        rx = roi_row['std_x']
        ry = roi_row['std_y']
        rw = roi_row['w_px']
        rh = roi_row['h_px']

        track_ids = tracks['track_id'].unique()

        for track_id in track_ids:
            track = tracks[tracks['track_id'] == track_id].sort_values('frame')

            inside_mask = (
                (track['center_x'] >= rx) & (track['center_x'] <= rx + rw) &
                (track['center_y'] >= ry) & (track['center_y'] <= ry + rh)
            )

            inside_frames = track[inside_mask]

            if len(inside_frames) == 0:
                continue

            entry_height_um  = inside_frames.iloc[0]['center_y'] / PIXELS_PER_MICRON
            exit_height_um   = inside_frames.iloc[-1]['center_y'] / PIXELS_PER_MICRON
            height_change_um = exit_height_um - entry_height_um

            results.append({
                'entry_height_um' : entry_height_um,
                'exit_height_um'  : exit_height_um,
                'height_change_um': height_change_um
            })

    return pd.DataFrame(results)


if SHOW_ROIS:
    results_df = calculate_entry_exit_heights(df, roi_df)
    print(f"Found {len(results_df)} ROI interaction events.")

    # --- BINNING & AGGREGATION ---
    results_df['bin'] = (results_df['entry_height_um'] // BIN_SIZE_UM) * BIN_SIZE_UM

    agg_table = results_df.groupby('bin').agg(
        N                  = ('entry_height_um',  'count'),
        mean_exit_height   = ('exit_height_um',   'mean'),
        mean_height_change = ('height_change_um', 'mean')
    ).round(4)

    # Exclude bins below minimum detection threshold
    agg_table.loc[agg_table['N'] < MIN_DETECTIONS, ['mean_exit_height', 'mean_height_change']] = np.nan

    # Fixed range 0 to 50um — empty bins stay as NaN for correct aggregation
    full_bin_range = np.arange(0.0, 50.0 + BIN_SIZE_UM, BIN_SIZE_UM)
    agg_table = agg_table.reindex(full_bin_range)

    agg_table.index.name = 'Entry Height Bin (um from Skin)'
    agg_table.columns    = ['N', 'Mean Exit Height (um)', 'Mean Height Change (um)']

    agg_table.to_csv(entry_exit_output_path)
    print(f"ROI entry/exit height analysis complete.")
    print(f"Saved: {entry_exit_output_path}")
    print(agg_table)

else:
    print("SHOW_ROIS is False — skipping ROI entry/exit height analysis.")

In [ ]:
# =============================================================================
# STEP 5F — SPEED AND STOCHASTICITY VS HEIGHT (ROI STRIP)
# =============================================================================
# For each ROI, extracts all particle detections within the x-extent of the
# ROI (the "ROI strip") and calculates mean speed and stochasticity binned
# by height above the skin in 2um bins. Results are pooled across all ROIs
# in the video and saved as a single per-video CSV.
# Empty bins stored as NaN to allow correct weighted aggregation across videos.
# NOTE: Requires SHOW_ROIS = True and a valid ROI_CSV in configuration.
# =============================================================================

strip_output_path = os.path.join(OUTPUT_DIR, f"{video_name}_strip_speed_stochasticity_vs_height.csv")

STRIP_BIN_SIZE_UM = 2  # 2um bins for fine height resolution

def calculate_strip_speed_stochasticity(tracks, rois):
    """
    For each ROI, filter tracks to the x-extent of the ROI strip and calculate
    mean speed and stochasticity (temporal velocity fluctuation) per height bin.
    Results are pooled across all ROIs.

    Parameters:
        tracks (pd.DataFrame): Standardised tracking data with vx_um_s, vy_um_s.
        rois (pd.DataFrame): Standardised ROI data with std_x, std_y, w_px, h_px.

    Returns:
        pd.DataFrame: Pooled detections with height_um and speed_um_s columns.
    """
    all_strip_data = []

    for _, roi_row in rois.iterrows():
        rx = roi_row['std_x']
        rw = roi_row['w_px']

        # Filter to particles within the x-extent of this ROI strip
        strip_mask = (
            (tracks['center_x'] >= rx) &
            (tracks['center_x'] <= rx + rw)
        )
        strip_data = tracks[strip_mask].copy()

        if strip_data.empty:
            continue

        strip_data['height_um'] = strip_data['center_y'] / PIXELS_PER_MICRON
        all_strip_data.append(strip_data)

    if not all_strip_data:
        return pd.DataFrame()

    return pd.concat(all_strip_data, ignore_index=True)


if SHOW_ROIS:
    # Ensure velocity components exist
    if 'speed_um_s' not in df.columns:
        df['speed_um_s'] = np.sqrt(df['vx_um_s']**2 + df['vy_um_s']**2)

    # Cap implausible speeds
    df_strip = df[df['speed_um_s'] <= MAX_SPEED_UM_S].copy()

    strip_df = calculate_strip_speed_stochasticity(df_strip, roi_df)

    if strip_df.empty:
        print("No strip data found — check ROI coordinates.")
    else:
        # Bin by height in 2um bins
        strip_df['bin'] = (strip_df['height_um'] // STRIP_BIN_SIZE_UM) * STRIP_BIN_SIZE_UM

        # Per bin — compute mean speed and stochasticity
        # Stochasticity = std of frame-mean speed across frames (consistent with 5A)
        def bin_stats(group):
            frame_mean_speed = group.groupby('frame')['speed_um_s'].mean()
            return pd.Series({
                'N': len(group),
                'Avg Speed (um/s)': group['speed_um_s'].mean(),
                'Stochasticity (um/s)': frame_mean_speed.std()
            })

        strip_table = strip_df.groupby('bin').apply(
            bin_stats, include_groups=False
        )

        # Exclude bins below minimum detection threshold
        strip_table.loc[strip_table['N'] < MIN_DETECTIONS,
                        ['Avg Speed (um/s)', 'Stochasticity (um/s)']] = np.nan

        # Force full range 0 to 50um
        full_bin_range = np.arange(0.0, 55.0, STRIP_BIN_SIZE_UM)
        strip_table = strip_table.reindex(full_bin_range)

        # Reorder columns — N first after index, consistent with other outputs
        strip_table = strip_table[['N', 'Avg Speed (um/s)', 'Stochasticity (um/s)']]
        strip_table.index.name = 'Height Bin (um from Skin)'
        strip_table = strip_table.round(4)

        strip_table.to_csv(strip_output_path)
        print(f"Strip speed/stochasticity analysis complete.")
        print(f"Saved: {strip_output_path}")
        print(strip_table)

else:
    print("SHOW_ROIS is False — skipping strip speed/stochasticity analysis.")

## Pipeline Complete

All outputs have been saved to your specified `OUTPUT_DIR`.
This pipeline is under active development — additional analysis cells will be added in future versions.